# 06 可选：Grad-CAM 与错误案例分析

完成 05 后，对验证集选定的最终 E3 模型分析正确和错误样本。安装 `requirements-demo.txt` 后再开启 `RUN_ANALYSIS`。模型和预处理从八分类 checkpoint 加载，不手工填写历史实验目录。

Grad-CAM 展示类别得分敏感区域；它不能证明模型理解病斑机理，也不能证明病害标注正确。少量案例用于解释，不能代替整体评价。


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        for root in (candidate, candidate / "plant-disease-project"):
            if (root / "configs" / "base.yaml").is_file():
                return root
    raise FileNotFoundError("请在项目根目录或 notebooks 目录打开本 Notebook。")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_config, load_labels

BASE = load_config(PROJECT_ROOT / "configs" / "base.yaml")
EXPERIMENTS = load_config(PROJECT_ROOT / "configs" / "experiments.yaml")
LABELS = load_config(PROJECT_ROOT / "configs" / "labels.yaml")
CLASS_NAMES = [item["label"] for item in load_labels(PROJECT_ROOT)]
assert len(CLASS_NAMES) == 8 and len(set(CLASS_NAMES)) == 8
assert [item["label_index"] for item in LABELS["classes"]] == list(range(8))
print("项目目录：", PROJECT_ROOT)
print("类别数量：", len(CLASS_NAMES), "；随机种子：", BASE["seed"])


## 1. 加载真实案例

默认每种正确性展示两个案例；正式报告应说明选例规则，并覆盖不同类别和场景。可将 `ANALYSIS_DATASET` 改为 `plantvillage_test` 比较标准背景。


In [ ]:
from src.experiments import selected_run

RUN_ANALYSIS = False
ANALYSIS_DATASET = "plantdoc_external_all"
MAX_PER_GROUP = 2
analysis_run_dir = None
selected_cases = []
try:
    analysis_run_dir = selected_run(PROJECT_ROOT, "E3")
except (ValueError, FileNotFoundError) as error:
    print("请先完成 02—05：", error)

if RUN_ANALYSIS and analysis_run_dir is not None:
    prediction_path = analysis_run_dir / "predictions.csv"
    if not prediction_path.is_file():
        print("缺少逐图预测，请先运行 05 的评价单元格。")
    else:
        predictions = pd.read_csv(prediction_path)
        subset = predictions[predictions["dataset"] == ANALYSIS_DATASET].copy()
        subset["is_correct"] = subset["true_label"] == subset["predicted_label"]
        for correct in (True, False):
            group = subset[subset["is_correct"] == correct]
            selected_cases.extend(group.head(MAX_PER_GROUP).to_dict("records"))
            print("正确案例" if correct else "错误案例", "总数：", len(group))
        display(pd.DataFrame(selected_cases))
else:
    print("分析关闭：完成最终评价后将 RUN_ANALYSIS 设为 True。")


## 2. 生成热力图

以预测类别作为解释目标，输出写入当前 run_id 对应目录。确定性预处理与模型评价一致；原图和热力图并排显示。


In [ ]:
if RUN_ANALYSIS and analysis_run_dir is not None and selected_cases:
    from PIL import Image
    from src.data import build_transforms
    from src.evaluate import load_trained_model
    from src.explain import gradcam_image
    from src.utils import get_device

    device = get_device(BASE.get("device", "auto"))
    model, checkpoint = load_trained_model(analysis_run_dir, device=device)
    data_config = checkpoint["config"]["data"]
    transform = build_transforms("none", **{key: data_config[key]
                                 for key in ("image_size", "mean", "std") if key in data_config})
    output_dir = PROJECT_ROOT / "outputs" / "figures" / analysis_run_dir.name / "gradcam"
    output_dir.mkdir(parents=True, exist_ok=True)
    for index, case in enumerate(selected_cases, start=1):
        image_path = PROJECT_ROOT / case["image_path"]
        with Image.open(image_path) as source:
            image = source.convert("RGB")
        overlay = gradcam_image(model, checkpoint["model_name"], image, transform,
                                target_class=int(case["predicted_index"]), device=device)
        overlay.save(output_dir / f"{ANALYSIS_DATASET}_{index:02d}.png")
        print("原图：", case["image_path"])
        print("真实：", case["true_label"], "预测：", case["predicted_label"],
              "模型概率：", round(float(case["confidence"]), 4))
        display(image.resize((224, 224)), overlay)
else:
    print("尚未生成热力图。")


## 分析记录（实际生成后填写）

| 案例路径 | 真实 / 预测类别 | 场景、病斑特点 | 注意区域 | 可能错误原因 |
| --- | --- | --- | --- | --- |

记录选择规则、类别覆盖、是否关注背景及解释限制。对源数据标签的怀疑需另行核实，不能直接用模型结果改标。
